# **[실습 2] 프롬프트 엔지니어링**

### **Q1. Few-Shot**

고객 문의를 분류를 few-shot으로 제공하는 프롬프트를 작성하고, 새로운 문의 5건을 분류시킨다.

**라벨**

- 결제/주문
- 배송/품질
- 계정

**분류 대상**

- "환불 신청했는데 언제 처리되나요?"<br>
- "비밀번호를 잊어버렸어요"<br>
- "배송 조회가 안 돼요"<br>
- "제품 사용법 설명서가 없어요"<br>
- "쿠폰이 적용되지 않아요"<br>

**출력 포맷**

JSON

[<br>
      {"문의": "환불 신청했는데 언제 처리되나요?", "라벨": "..."}, <br>
      {"문의": "비밀번호를 잊어버렸어요", "라벨": "..."} <br>
]

In [ ]:
# 프롬프트 엔지니어링 (prompt engineering)
# AI에게 결과물을 내가 원하는 형태 또는 방식대로 response를 받을수 있도록 설계하는 기술
# instruction, temperature, max_output_tokens -> prompt engineering의 일부

# 프롬프트 엔지니어링은 크게 3 단위로 나뉜다.
# 1. system prompt
# -> 사실상 service proivder가 지정해둔 규칙을 담은 instruction
# -> 보안상의 문제라던가, 사회적인 문제라던가 그런 문제를 일으키지 않도록 함
# -> 일반 유저들이 접근하거나 보거나 수정하지 못하도록 되어있는 규제

# 2. developer prompt
# -> 어플리케이션의 동작/행동을 지정한다. 
# -> instruction..이런것들이 다 여기에 해당된다.
# -> role/persona등등.. 기존 모델에 decorator처럼 씌워주는것.

# 3. user prompt
# -> 저희가 넣는 input (prompt)
# -> 내가 AI에게 받고싶은 응답에 대한 질문을 넣는것.

# 프롬프트의 동작
# 1. prompt가 모델에 입력이 됩니다. (해석-> encoder 쪽에 들어가서 tokenizing, embeding..)
# 2. 모델이 응답을 "생성"하는 과정 (decoder) 
#    attention + skipgram model -> 다음 단어가 이번 질문의 답으로 계속 학습
# -> LLM은 응답 생성 -> 이전 단어들로 다음 단어 때려맞추기를 하고 있다.
# 그렇기 때문에 앞서 들어가있는 내용이 -> 이후 내용에 영향을 줄 수 있다.

# 두번째로 쓰는 이유 (요즘 모델은 사실 잘 없다..) -> 환각 현상을 방지하기 위해 쓴다.
# 요즘은 안보이는데 -> gpt같은경우는 tool call이라는 형태로 방지
# 1. 프롬프트 엔지니어링으로 해결 -> 모르는건 모른다고 해라 왜 아는 척 하냐
# 2. RAG : 모르는 정보에 대해서 우리가 주입시켜주고 답을 하게 하는 형태
# 3. tool call : web search라는 툴을 달아주면 -> 알아서 fact check를 하고 응답

# 예시기반 프롬프팅
# 1. zero-shot prompt : 여러분들이 지금까지 하고 있던 프롬프트 방식
#    아무 정보도 주어지지 않고, 지금 gpt가 가진 지식만으로 답하게 하는 것.
# 2. few-shot prompt : 예시를 몇가지를 주어주고, 이걸 학습해서 이대로 응답을 내라
#    몇가지의 예시를 주어준다. (1개를 주어주면 one-shot이라고도 한다.)
# 3. counter-example prompt : 좋은 예시와 나쁜 예시를 번갈아가변서 보여주고 학습을 시키는 방법

# 추론기반 (생각하는 방법을 알려준다)
# 1. CoT (Chain of Thought)
#    그냥 답만 내뱉지 말고 너가 왜 이런 결론을 냈는지에 대해서 설명을 해라
#    -> 단순하게 "단계별로 설명해줘", "let's think step by step"
# 2. self-consistency 
#    ML -> voting model
#    여러번 temperature를 바꿔가면서 얘한테 똑같은 추론과정이나 응답을 요청
#    지금까지 나온 정답들을 다시 다 넣고, 이것을 기반으로 최종 정답을 내달라

# 검증기반 
# 1. prompt chaining : 프롬프트를 단계별로 쪼개서 이 전의 정답을 다음으로 넘기면서
#    순차적인 응답을 받는 기법
# agent에서 예를들어서 : 이런 게임만들어줘 / 1. 이런 UI 2. 이런 기능 3. 캐릭터를 4. 때리는로직을 추가한다.
# 젠슨황의 사과문을 써야 한다.
# 1단계 사과를 박는다.
# 2단계 그럴싸한 자초지종을 설명한다
# 3단계 모두가 나를 불쌍하게 여길만한 시나리오를 만들어낸다.
# 4단계 다신 이런일이 일어나지 않겠다라는 말을 한다. 

# output format

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# API KEY 가져오기
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

response = client.responses.create(
    model='gpt-4o-mini',

    # zero shot 예시
    # input=""""
    # 이 다음 리뷰의 감정분석을 해줘.
    # "이 영화 겁나 재미없네."
    # """

    # one shot 예시
    # input ="""
    # 아래는 몇가지 제품 리뷰에 대한 감정 분석의 예시이다.

    # [예시] 
    # "너무너무 맘에 들어요" -> 긍정
    # "우리 아이가 너무 좋아해요" -> 긍정
    # "제품을 인도네시아에서 만들어서 오나요?" -> 부정
    # "와 오자말자 팔다리가 모두 분리가 되네요!" -> 부정

    # 다음 제품 리뷰에 대한 감정 분석을 해줘
    # 1. "와 진짜 별로네."
    # 2. "이거 원래 색깔이 이런가요?"
    # 3. "내가 주문한게 아니예요?"
    # """
    
    # counter-example
    # input='''
    # 자 너는 젠슨황의 불륜 사태에 대해 사과문을 대신 올려주는 봇이야. 
    # 너가 사과문을 올릴때 이런건 주의해야 해. 

    # 나쁜 예시 : 저를 믿고 따라주신 모든 분들에게 심려를 끼쳐드려 죄송합니다.
    # 좋은 예시 : 아 ㅈㅅㅈㅅ

    # 나쁜 예시 : 깊게 반성하고 있습니다.
    # 좋은 예시 : 아 사람이 그럴수도 있지 뭐

    # 자 이제 사과문을 써줘
    # '''

    # CoT
    # input='''
    # 자 지금 상자 안에는 사과가 10개가 들어있어.
    # 이런 상자가 지금 5개가 있어.
    # 여기서 혜리양이 각 상자에서 사과를 하나씩 빼서 먹었어.
    # 그리고 한 상자에서 다른 상자에 비어있는 칸들을 채워서 숨기기 위해 하나씩 빼서 다시 채워넣었어.
    # 그렇다면 꽉 차있지 않은 사과의 개수는 몇개일까
    # 자 그럼 총 사과는 몇개인가? 
    # 이제 이 과정을 단계적으로 펼쳐서 계산 과정을 보여주도록 해
    # '''
) 
print(response.output_text)


In [3]:
# code here
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# API KEY 가져오기
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)


     # one shot 예시
instructions = """
너는 경력이 많은 고객 문의 데이터 분석 전문가이다. 
문의글에 대해 어떤 분야에 글인지 판단하고 데이터를 수집하는 것이 너의 일이다.
라벨은 "결제/주문","배송/품질", "계정"이 있다.

그리고 각 라벨에 대한 예시는 아래와 같다.
    
[결제/주문]
1. 카드로 결제했는데 주문 내역에는 아무것도 나오지 않아요. 주문이 정상적으로 접수된 건가요?
2. 같은 상품이 두 번 결제됐어요. 중복 결제된 금액을 취소해 주세요.
3. 주문할 때 할인 쿠폰을 적용하지 못했는데, 결제 후에도 적용할 수 있나요?
4. 방금 주문한 상품의 색상을 변경하고 싶은데 어떻게 해야 하나요?
5. 주문을 취소했는데 아직 환불이 안 됐어요. 언제 입금되나요?

[배송/품질]
1. 배송 예정일이 지났는데 상품이 아직 도착하지 않았어요. 언제 받을 수 있나요?
2. 배송 완료로 표시되는데 택배를 받지 못했어요. 확인 부탁드립니다.
3. 주문한 상품과 다른 상품이 도착했어요. 교환은 어떻게 신청하나요?
4. 상품을 받았는데 표면에 흠집이 있고 일부가 깨져 있어요. 새 상품으로 교환해 주세요.
5. 옷을 한 번 세탁했는데 봉제선이 풀어졌어요. 불량으로 반품할 수 있나요?

[계정]
1. 비밀번호를 잊어버렸는데 재설정 이메일이 오지 않아요. 어떻게 해야 하나요?
2. 휴대전화 번호가 바뀌어서 로그인 인증을 받을 수 없어요. 다른 인증 방법이 있나요?
3. 회원 탈퇴를 하고 싶은데 어디에서 신청할 수 있나요?
4. 제가 로그인하지 않은 시간에 로그인 알림이 왔어요. 계정이 도용된 건가요?
5. 이메일로 가입한 계정과 카카오로 가입한 계정을 하나로 통합할 수 있나요?

출력 형식은 JSON 파일로 아래와 같이 출력하라.
JSON
[
    {"문의": "환불 신청했는데 언제 처리되나요?", "라벨": "..."},
    {"문의": "비밀번호를 잊어버렸어요", "라벨": "..."} 
]
    
"""
prompt="""
다음 문의에 대해 "결제/주문","배송/품질", "계정" 중 어떤 분야의 글인지 분류하라. 
    
"환불 신청했는데 언제 처리되나요?"
"비밀번호를 잊어버렸어요"
"배송 조회가 안 돼요"
"제품 사용법 설명서가 없어요"
"쿠폰이 적용되지 않아요"
"너무 제품을 이상하게 포장하신거 아니에요? 제품 빨리 반품해주시고 환불도 해주세요."
"""

response = client.responses.create(
    model='gpt-4o-mini',
    input = prompt,
    instructions=instructions
)
print(response.output_text)

```json
[
    {"문의": "환불 신청했는데 언제 처리되나요?", "라벨": "결제/주문"},
    {"문의": "비밀번호를 잊어버렸어요", "라벨": "계정"},
    {"문의": "배송 조회가 안 돼요", "라벨": "배송/품질"},
    {"문의": "제품 사용법 설명서가 없어요", "라벨": "배송/품질"},
    {"문의": "쿠폰이 적용되지 않아요", "라벨": "결제/주문"},
    {"문의": "너무 제품을 이상하게 포장하신거 아니에요? 제품 빨리 반품해주시고 환불도 해주세요.", "라벨": "배송/품질"}
]
```


### **Q2. Counter-Example**

"공손한 문의 답장" 프롬프트를 counter-example을 사용해서 만들어 주세요.
- 반례들을 제공해서 문제점이 무엇인지 스스로 판단하게 합니다.

**목표/상황**

다음과 같은 컴플레인 메일이 왔을 때, 답장을 생성한다.

- "제품이 3주째 배송 안 됨. 너무 화남"

**출력 포맷**

제목: [이메일 제목]

본문:
- 인사
- 사과 및 지연 사유
- 구체적 조치
- 마무리 인사

담당자: [가상의 이름]

In [ ]:
# code here

### **Q3. CoT**

단계별 추론을 활용하여 수학 문제를 풀게 하시오.

**문제**

"한 카페에서 아메리카노는 4,500원, 라떼는 5,000원이다. <br>
오늘 아메리카노 37잔, 라떼 24잔이 팔렸고, 재료비는 매출의 30%였다. <br>
오늘의 순이익(매출-재료비)은 얼마인가?"

**출력 포맷**

[1단계] 아메리카노 매출 계산: ...<br>
[2단계] 라떼 매출 계산: ...<br>
[3단계] 총 매출 계산: ...<br>
[4단계] 재료비 계산: ...<br>
[5단계] 순이익 계산: ...<br>

[최종 응답] : XXX원

In [ ]:
# code here